# Persistent Person Tracking

Track one user-selected person through temporary Track ID changes, foreground
occlusions, and scene cuts. Temporary tracker IDs provide motion continuity;
an explicit face gallery represents the persistent `TARGET_PERSON` identity.

This notebook writes to its own output directory and does not modify the
baseline `subject_reframe.ipynb` workflow.


## 1. Configuration

Edit this section before running the notebook. The defaults favor safe identity
decisions: automatic gallery expansion is disabled and uncertain scenes ask for
manual confirmation rather than guessing.


In [ ]:
from pathlib import Path

# ── Project and input ──────────────────────────────────────────────────────
# The notebook works from either project/ or project/notebooks/.
WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
# Video to process. MP4/H.264 is the most reliable input for OpenCV.
INPUT_VIDEO = PROJECT_ROOT / "input" / "source.mp4"
# First source timestamp to process, in seconds.
PROCESS_START_SECONDS = 5.0
# Final source timestamp; -1 processes through the end of the video.
PROCESS_END_SECONDS = 20.0

# ── Independent persistent-tracking output ────────────────────────────────
OUTPUT_DIRECTORY = PROJECT_ROOT / "output" / "persistent_person_tracking"
TRACK_CACHE_DIRECTORY = OUTPUT_DIRECTORY / "cache" / "tracking"
IDENTITY_CACHE_DIRECTORY = OUTPUT_DIRECTORY / "cache" / "identity"
MODEL_DIRECTORY = PROJECT_ROOT / "models"
GALLERY_ARRAY_FILE = OUTPUT_DIRECTORY / "target_gallery.npz"
GALLERY_METADATA_FILE = OUTPUT_DIRECTORY / "target_gallery.json"
IDENTITY_EVENTS_FILE = OUTPUT_DIRECTORY / "identity_events.json"
SCENE_ACTIONS_FILE = OUTPUT_DIRECTORY / "scene_actions.json"
SILENT_VIDEO = OUTPUT_DIRECTORY / "persistent_person_silent.mp4"
FINAL_VIDEO = OUTPUT_DIRECTORY / "persistent_person_reel.mp4"
BROWSER_PREVIEW_VIDEO = OUTPUT_DIRECTORY / "persistent_person_preview.mp4"
PREVIEW_IMAGE = OUTPUT_DIRECTORY / "preview.jpg"

# ── Development and cache behavior ────────────────────────────────────────
# True always reruns scene detection; False reuses matching scene metadata.
FORCE_SCENE_REDETECTION = False
# True reruns YOLO tracking for every scene.
FORCE_RETRACK = True
# True reruns face identity resolution instead of loading matching caches.
FORCE_IDENTITY_RESOLUTION = True
# True loads the previously saved biometric gallery. Leave False when changing videos or targets.
REUSE_SAVED_GALLERY = False
# True pauses to approve even automatically resolved scenes.
REVIEW_AUTOMATIC_RESOLUTION = False
# True downloads the official YuNet/SFace ONNX files if they are absent.
DOWNLOAD_FACE_MODELS = True

# ── Scene detection ────────────────────────────────────────────────────────
AUTOMATIC_SCENE_DETECTION = True
# Known cut timestamps may be added manually, for example [12.4, 28.1].
MANUAL_SCENE_CUTS = []
MINIMUM_SCENE_SECONDS = 0.5
ADAPTIVE_SCENE_THRESHOLD = 5.0
ADAPTIVE_MIN_CONTENT_VALUE = 7.0
ADAPTIVE_WINDOW_WIDTH = 2
ENABLE_HARD_CUT_DETECTOR = True
HARD_CUT_THRESHOLD = 15.0
ENABLE_FADE_DETECTOR = True
FADE_BRIGHTNESS_THRESHOLD = 12.0
SCENE_FRAME_SKIP = 0
SCENES_PER_PREVIEW_PAGE = 9
SCENE_PREVIEW_COLUMNS = 3

# ── Temporary person tracking ──────────────────────────────────────────────
# The small model is faster; use yolo11s.pt or yolo11m.pt for difficult footage.
MODEL_NAME = "yolo11m.pt"
# Appearance ReID plus a longer lost-track buffer helps short occlusions.
TRACKER_NAME = str(PROJECT_ROOT / "config" / "subject_deepocsort_reid.yaml")
TRACKING_QUANTIZE = 16
# Must be a multiple of 32. Raise to 640 for small or distant people.
TRACKING_IMAGE_SIZE = 576
# Analyze every Nth source frame. Use 1 for maximum identity stability.
TRACKING_FRAME_STRIDE = 2
DETECTION_CONFIDENCE = 0.20
TRACKING_IOU = 0.50
# Maximum person detections retained per frame; raise only for crowded footage.
MAX_PERSON_DETECTIONS = 20

# ── Face recognition and persistent identity ──────────────────────────────
FACE_DETECTION_THRESHOLD = 0.80
MINIMUM_FACE_PIXELS = 36
MINIMUM_FACE_QUALITY = 0.35
# Reject a detected face outside the central upper region of its person box.
# This prevents an overlapping neighbour's face from entering the gallery.
MINIMUM_HEAD_CENTER_X_RATIO = 0.12
MAXIMUM_HEAD_CENTER_X_RATIO = 0.88
MAXIMUM_HEAD_CENTER_Y_RATIO = 0.48
# Verify periodically; overlaps and lost IDs trigger immediate checks.
FACE_CHECK_SECONDS = 0.75
# Even during overlap or ID loss, limit expensive face checks to this cadence.
URGENT_FACE_CHECK_SECONDS = 0.20
# Search this many seconds near a new scene's beginning before asking the user.
SCENE_IDENTITY_SEARCH_SECONDS = 12.0
# Check this often while looking for an automatic scene anchor.
AUTOMATIC_ANCHOR_CHECK_SECONDS = 0.25
# Require this many consistent face matches before a scene is auto-selected.
AUTOMATIC_ANCHOR_CONFIRMATIONS = 2
# Never trust motion-only identity for longer than this.
MAXIMUM_UNVERIFIED_SECONDS = 2.0
# SFace cosine scores are model-specific scores, not probabilities.
FACE_MATCH_THRESHOLD = 0.42
FACE_MISMATCH_THRESHOLD = 0.24
# Reopen manual review after this many consecutive visible-face mismatches.
# Missing/back-facing faces do not count as mismatches.
FACE_MISMATCH_CONFIRMATIONS = 2
# Best face match must exceed the second-best candidate by this amount.
FACE_AMBIGUITY_MARGIN = 0.08
OVERLAP_IOU_THRESHOLD = 0.18
MAXIMUM_CENTER_JUMP = 0.30
# Maximum trajectory prediction error, normalized by the previous body diagonal.
TRAJECTORY_MAXIMUM_ERROR = 0.45
# Ask about another Track ID only when its trajectory is better by this margin.
TRAJECTORY_SWITCH_MARGIN = 0.10
# Higher values retain more of the established velocity and resist sudden branch changes.
TRAJECTORY_VELOCITY_SMOOTHING = 0.65
# A manual click remains authoritative for at least this long around its frame.
MANUAL_ANCHOR_LOCK_SECONDS = 0.75
# Maximum trusted target faces retained in the local gallery.
MAXIMUM_GALLERY_ENTRIES = 24
# Similar samples above this value are not duplicated in the gallery.
GALLERY_DIVERSITY_SIMILARITY = 0.95
# Prevent one scene or repeated corrections from dominating the gallery.
MAXIMUM_GALLERY_ENTRIES_PER_SCENE = 4
# A face gathered around a selected Track ID must resemble the clicked anchor face.
BOOTSTRAP_FACE_CONSISTENCY_THRESHOLD = 0.35
# If the clicked frame has no usable face, search this many seconds in the same scene.
MANUAL_FACE_SEARCH_SECONDS = 3.0
# Existing galleries may auto-accept only a coherent batch this strong.
# The first gallery seed is always shown to the user for confirmation.
GALLERY_AUTO_ACCEPT_MINIMUM_FACES = 2
GALLERY_AUTO_ACCEPT_MINIMUM_QUALITY = 0.75
GALLERY_AUTO_ACCEPT_SIMILARITY = 0.50

# ── Manual identity review ─────────────────────────────────────────────────
SELECTION_FRAME_OVERRIDES = {}
TRACK_PREVIEW_CARD_WIDTH = 240
TRACK_PREVIEW_CARD_HEIGHT = 300
# Each click confirms one visible fragment. If another gap remains, Section 7 asks again.

# ── Portrait crop and render ───────────────────────────────────────────────
RENDER_QUALITY = "fast"  # "fast" = 540×960; "high" = 1080×1920.
DRAFT_OUTPUT_WIDTH = 540
DRAFT_OUTPUT_HEIGHT = 960
OUTPUT_WIDTH = 1080
OUTPUT_HEIGHT = 1920
BROWSER_PREVIEW_WIDTH = 360
HORIZONTAL_MARGIN = 0.25
TOP_MARGIN = 0.18
BOTTOM_MARGIN = 0.22
SMOOTHING_SECONDS = 0.45
MAX_INTERPOLATION_SECONDS = 0.60
LOW_CONFIDENCE_THRESHOLD = 0.35
# Fast mode always uses current-frame average-colour fill.
PADDING_MODE = "blur"
# Short unresolved gaps retain the reliable target crop; long gaps block rendering for review.
MISSING_TRACK_POLICY = "hold"
ABSENT_SCENE_POLICY = "fit"

for directory in (OUTPUT_DIRECTORY, TRACK_CACHE_DIRECTORY, IDENTITY_CACHE_DIRECTORY, MODEL_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Input video:", INPUT_VIDEO)
print("Persistent output:", OUTPUT_DIRECTORY)


## 2. Load the reusable project code


In [ ]:
import sys

SRC_DIRECTORY = PROJECT_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from subject_reframe import (
    CropConfig, FaceConfig, FaceRecognizer, IdentityResolver,
    PERSISTENT_TARGET_TRACK_ID, PersonTrackingSession, ResolverConfig,
    Scene, SKIP_SCENE, TargetIdentity, bootstrap_target_identity,
    build_crop_plan, build_identity_cache_metadata, build_tracking_cache_metadata,
    collect_target_face_candidates,
    select_coherent_face_cluster,
    create_browser_preview, detect_scenes, ensure_face_models, ffmpeg_available,
    inspect_video, load_identity_resolution, load_tracking_records,
    manual_resolution_from_tracks, merge_resolved_records, mux_original_audio,
    print_identity_summary, print_scene_table, print_warning_summary,
    render_video, resolve_processing_range, save_identity_events,
    save_identity_resolution, save_preview, save_report, save_tracking_records,
    select_scene_track_widget, show_identity_events, show_scene_midpoints,
    review_face_gallery_candidates_widget, show_target_gallery,
    source_segments_from_plans, supplement_resolution,
    unconfirmed_source_switches,
)
print("Subject Reframe package loaded from:", SRC_DIRECTORY)


## 3. Check Python, FFmpeg, and the RTX GPU


In [ ]:
import cv2
import torch
import ultralytics

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = 0 if CUDA_AVAILABLE else "cpu"
EFFECTIVE_TRACKING_QUANTIZE = TRACKING_QUANTIZE if CUDA_AVAILABLE else None
print("OpenCV:", cv2.__version__)
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(properties.total_memory / 1024**3, 2))
else:
    print("WARNING: YOLO tracking will run on CPU.")
print("OpenCV face APIs:", hasattr(cv2, "FaceDetectorYN") and hasattr(cv2, "FaceRecognizerSF"))
print("OpenCV CUDA devices:", cv2.cuda.getCudaEnabledDeviceCount() if hasattr(cv2, "cuda") else 0)
print("FFmpeg available:", ffmpeg_available())


## 4. Inspect the input and selected processing range


In [ ]:
if not INPUT_VIDEO.exists():
    raise FileNotFoundError(f"Place the source video at {INPUT_VIDEO}, or change INPUT_VIDEO in Section 1.")
video_info = inspect_video(INPUT_VIDEO)
processing_range = resolve_processing_range(video_info, PROCESS_START_SECONDS, PROCESS_END_SECONDS)
print(f"Resolution: {video_info.width} × {video_info.height}")
print(f"Frame rate: {video_info.fps:.3f} fps")
print(f"Duration: {video_info.duration_seconds:.2f} seconds")
print(
    f"Processing: {processing_range.start_seconds:.3f}s–{processing_range.end_seconds:.3f}s "
    f"({processing_range.frame_count:,} frames)"
)


## 5. Prepare YuNet and SFace

The face models run periodically on CPU in the standard `opencv-python` wheel;
YOLO continues using the RTX GPU. Model files stay local under `models/`.


In [ ]:
if DOWNLOAD_FACE_MODELS:
    FACE_DETECTOR_MODEL, FACE_RECOGNIZER_MODEL = ensure_face_models(MODEL_DIRECTORY)
else:
    FACE_DETECTOR_MODEL = MODEL_DIRECTORY / "face_detection_yunet_2023mar.onnx"
    FACE_RECOGNIZER_MODEL = MODEL_DIRECTORY / "face_recognition_sface_2021dec.onnx"

face_config = FaceConfig(
    detection_threshold=FACE_DETECTION_THRESHOLD,
    minimum_face_pixels=MINIMUM_FACE_PIXELS,
    minimum_quality=MINIMUM_FACE_QUALITY,
    minimum_head_center_x_ratio=MINIMUM_HEAD_CENTER_X_RATIO,
    maximum_head_center_x_ratio=MAXIMUM_HEAD_CENTER_X_RATIO,
    maximum_head_center_y_ratio=MAXIMUM_HEAD_CENTER_Y_RATIO,
)
face_recognizer = FaceRecognizer(FACE_DETECTOR_MODEL, FACE_RECOGNIZER_MODEL, face_config)
FACE_MODEL_METADATA = face_recognizer.model_metadata
print("YuNet:", FACE_DETECTOR_MODEL)
print("SFace:", FACE_RECOGNIZER_MODEL)


## 6. Detect and review scenes


In [ ]:
scene_cache = OUTPUT_DIRECTORY / "scene_detection.json"
if AUTOMATIC_SCENE_DETECTION:
    scenes = detect_scenes(
        INPUT_VIDEO,
        adaptive_threshold=ADAPTIVE_SCENE_THRESHOLD,
        min_content_value=ADAPTIVE_MIN_CONTENT_VALUE,
        window_width=ADAPTIVE_WINDOW_WIDTH,
        use_content_detector=ENABLE_HARD_CUT_DETECTOR,
        content_threshold=HARD_CUT_THRESHOLD,
        use_fade_detector=ENABLE_FADE_DETECTOR,
        fade_threshold=FADE_BRIGHTNESS_THRESHOLD,
        min_scene_seconds=MINIMUM_SCENE_SECONDS,
        frame_skip=SCENE_FRAME_SKIP,
        manual_cut_seconds=MANUAL_SCENE_CUTS,
        start_frame=processing_range.start_frame,
        end_frame=processing_range.end_frame,
        cache_path=scene_cache,
        force_recompute=FORCE_SCENE_REDETECTION,
    )
else:
    manual_frames = {int(round(value * video_info.fps)) for value in MANUAL_SCENE_CUTS}
    cut_frames = sorted({
        processing_range.start_frame, processing_range.end_frame,
        *(frame for frame in manual_frames if processing_range.start_frame < frame < processing_range.end_frame),
    })
    scenes = [
        Scene(index, start, end, video_info.fps)
        for index, (start, end) in enumerate(zip(cut_frames[:-1], cut_frames[1:]))
        if end > start
    ]
print_scene_table(scenes)
show_scene_midpoints(
    INPUT_VIDEO, scenes,
    scenes_per_page=SCENES_PER_PREVIEW_PAGE,
    columns=SCENE_PREVIEW_COLUMNS,
)


## 7. Track and resolve `TARGET_PERSON`, one scene at a time

The first usable manual selection proposes faces for the persistent gallery;
inspect and approve those face thumbnails before they become trusted. Later
scenes are resolved automatically only when face evidence is unambiguous.
Motion-only alerts show every visible track instead of hiding alternatives.


In [ ]:
import json
from IPython.display import HTML, display

resolver_config = ResolverConfig(
    face_check_seconds=FACE_CHECK_SECONDS,
    urgent_face_check_seconds=URGENT_FACE_CHECK_SECONDS,
    scene_search_seconds=SCENE_IDENTITY_SEARCH_SECONDS,
    automatic_anchor_check_seconds=AUTOMATIC_ANCHOR_CHECK_SECONDS,
    automatic_anchor_confirmations=AUTOMATIC_ANCHOR_CONFIRMATIONS,
    maximum_unverified_seconds=MAXIMUM_UNVERIFIED_SECONDS,
    face_match_threshold=FACE_MATCH_THRESHOLD,
    face_mismatch_threshold=FACE_MISMATCH_THRESHOLD,
    face_mismatch_confirmations=FACE_MISMATCH_CONFIRMATIONS,
    ambiguity_margin=FACE_AMBIGUITY_MARGIN,
    overlap_iou_threshold=OVERLAP_IOU_THRESHOLD,
    maximum_center_jump=MAXIMUM_CENTER_JUMP,
    trajectory_maximum_error=TRAJECTORY_MAXIMUM_ERROR,
    trajectory_switch_margin=TRAJECTORY_SWITCH_MARGIN,
    trajectory_velocity_smoothing=TRAJECTORY_VELOCITY_SMOOTHING,
    manual_anchor_lock_seconds=MANUAL_ANCHOR_LOCK_SECONDS,
    bootstrap_face_consistency_threshold=BOOTSTRAP_FACE_CONSISTENCY_THRESHOLD,
    bootstrap_window_seconds=MANUAL_FACE_SEARCH_SECONDS,
)

if REUSE_SAVED_GALLERY and GALLERY_ARRAY_FILE.exists() and GALLERY_METADATA_FILE.exists():
    target_identity = TargetIdentity.load(GALLERY_ARRAY_FILE, GALLERY_METADATA_FILE)
    print("Loaded target gallery:", target_identity.public_summary())
else:
    target_identity = TargetIdentity(
        maximum_entries=MAXIMUM_GALLERY_ENTRIES,
        diversity_similarity=GALLERY_DIVERSITY_SIMILARITY,
    )

tracking_metadata = build_tracking_cache_metadata(
    video_info, scenes, model_name=MODEL_NAME, tracker_name=TRACKER_NAME,
    quantize=EFFECTIVE_TRACKING_QUANTIZE, image_size=TRACKING_IMAGE_SIZE,
    frame_stride=TRACKING_FRAME_STRIDE, confidence=DETECTION_CONFIDENCE,
    iou=TRACKING_IOU, max_detections=MAX_PERSON_DETECTIONS,
)
tracking_records = {}
identity_results = {}
scene_actions = {}
tracking_session = None


def get_tracking_session():
    global tracking_session
    if tracking_session is None:
        tracking_session = PersonTrackingSession(
            model_name=MODEL_NAME, tracker_name=TRACKER_NAME,
            device=DEVICE, quantize=EFFECTIVE_TRACKING_QUANTIZE,
            image_size=TRACKING_IMAGE_SIZE, frame_stride=TRACKING_FRAME_STRIDE,
            confidence=DETECTION_CONFIDENCE, iou=TRACKING_IOU,
            max_detections=MAX_PERSON_DETECTIONS,
        )
    return tracking_session


ACCEPT_OCCLUSION = "__accept_occlusion__"


def long_unresolved_ranges(scene, result):
    if result is None:
        return []
    return [
        (start, end)
        for start, end in result.unresolved_ranges
        if (end - start) / scene.fps > MAXIMUM_UNVERIFIED_SECONDS
    ]


async def request_manual_anchor(
    scene, scene_records, scene_position, *,
    allow_occlusion=False, candidate_track_ids=None,
):
    reviewed_frames = {}
    while True:
        action, preview_frame, visible_ids = await select_scene_track_widget(
            INPUT_VIDEO, scene, scene_records,
            override_seconds=SELECTION_FRAME_OVERRIDES,
            card_width=TRACK_PREVIEW_CARD_WIDTH,
            card_image_height=TRACK_PREVIEW_CARD_HEIGHT,
            allow_occlusion=allow_occlusion,
            allowed_track_ids=candidate_track_ids,
        )
        for track_id in visible_ids:
            reviewed_frames[int(track_id)] = preview_frame
        if isinstance(action, tuple) and action[0] == "time":
            SELECTION_FRAME_OVERRIDES[scene.index] = action[1]
            continue
        if action == "absent":
            return None, reviewed_frames
        if action == "skip":
            return SKIP_SCENE, reviewed_frames
        if allow_occlusion and action == "occluded":
            return ACCEPT_OCCLUSION, reviewed_frames
        if isinstance(action, int):
            return [action], reviewed_frames


try:
    for scene_position, scene in enumerate(scenes, start=1):
        display(
            HTML(
                f"<h3 style='margin:18px 0 4px'>Processing scene {scene.index} "
                f"({scene_position} of {len(scenes)})</h3>"
            )
        )
        scene_tracking_metadata = build_tracking_cache_metadata(
            video_info, [scene], model_name=MODEL_NAME, tracker_name=TRACKER_NAME,
            quantize=EFFECTIVE_TRACKING_QUANTIZE, image_size=TRACKING_IMAGE_SIZE,
            frame_stride=TRACKING_FRAME_STRIDE, confidence=DETECTION_CONFIDENCE,
            iou=TRACKING_IOU, max_detections=MAX_PERSON_DETECTIONS,
        )
        tracking_cache = TRACK_CACHE_DIRECTORY / f"scene_{scene.index:03d}.json"
        scene_records = None
        if tracking_cache.exists() and not FORCE_RETRACK:
            try:
                scene_records = load_tracking_records(
                    tracking_cache, expected_metadata=scene_tracking_metadata
                )
                print("Loaded temporary tracking cache.")
            except (ValueError, KeyError, TypeError) as error:
                print("Ignoring stale temporary tracking cache:", error)
        if scene_records is None:
            scene_records = get_tracking_session().track_scene(
                INPUT_VIDEO, scene, progress=True,
                progress_description=f"Tracking {scene_position}/{len(scenes)}",
            )
            save_tracking_records(scene_records, tracking_cache, metadata=scene_tracking_metadata)
        tracking_records.update(scene_records)

        result = None
        if target_identity.ready:
            identity_metadata = build_identity_cache_metadata(
                video_info, scene, tracking_metadata=scene_tracking_metadata,
                face_model_metadata=FACE_MODEL_METADATA,
                resolver_config=resolver_config,
                gallery_fingerprint=target_identity.fingerprint,
            )
            identity_cache = IDENTITY_CACHE_DIRECTORY / f"scene_{scene.index:03d}.json"
            if identity_cache.exists() and not FORCE_IDENTITY_RESOLUTION:
                try:
                    result = load_identity_resolution(
                        identity_cache, expected_metadata=identity_metadata
                    )
                    print("Loaded persistent identity cache.")
                except (ValueError, KeyError, TypeError) as error:
                    print("Ignoring stale persistent identity cache:", error)
            if result is None:
                result = IdentityResolver(
                    face_recognizer, target_identity, resolver_config
                ).resolve_scene(INPUT_VIDEO, scene, scene_records, progress=True)

        manual_required = result is None or result.status != "resolved"
        if result is not None and result.status == "resolved":
            print_identity_summary(scene, result)
            if REVIEW_AUTOMATIC_RESOLUTION:
                show_identity_events(INPUT_VIDEO, scene, result, scene_records)
                answer = input("Accept automatic identity? [Y/m/a/s]: ").strip().lower()
                if answer in {"m", "manual"}:
                    manual_required = True
                elif answer in {"a", "absent"}:
                    scene_actions[scene.index] = None
                    result = None
                elif answer in {"s", "skip"}:
                    scene_actions[scene.index] = SKIP_SCENE
                    result = None

        while manual_required:
            handoff_candidate_ids = None
            if result is not None:
                print_identity_summary(scene, result)
                show_identity_events(INPUT_VIDEO, scene, result, scene_records)
                pending_identity_reviews = [
                    event
                    for event in result.events
                    if event.event in {
                        "reassignment_requires_confirmation",
                        "current_track_requires_confirmation",
                    }
                ]
                unconfirmed_boundaries = unconfirmed_source_switches(
                    result,
                    maximum_confirmation_lead_frames=TRACKING_FRAME_STRIDE,
                )
                unsafe_ranges = long_unresolved_ranges(scene, result)
                if pending_identity_reviews:
                    handoff = min(
                        pending_identity_reviews, key=lambda event: event.frame
                    )
                    SELECTION_FRAME_OVERRIDES[scene.index] = handoff.frame / scene.fps
                    if handoff.event == "current_track_requires_confirmation":
                        print(
                            f"Repeated visible-face mismatch at "
                            f"{handoff.frame / scene.fps:.2f}s on Track "
                            f"{handoff.source_track_id}. Tracking is paused without "
                            "switching; all visible tracks are shown for confirmation."
                        )
                    elif (
                        handoff.face_similarity is not None
                        and handoff.face_similarity >= FACE_MATCH_THRESHOLD
                    ):
                        print(
                            f"Possible face-supported handoff at "
                            f"{handoff.frame / scene.fps:.2f}s. "
                            "Every visible track is shown for confirmation."
                        )
                    else:
                        score_text = (
                            f" Face score {handoff.face_similarity:.3f} did not meet "
                            f"the {FACE_MATCH_THRESHOLD:.2f} threshold."
                            if handoff.face_similarity is not None
                            else " No usable face supported the suggestion."
                        )
                        print(
                            f"Motion-only alert at {handoff.frame / scene.fps:.2f}s."
                            + score_text
                            + " The suggested Track ID is not trusted; all visible "
                            "tracks are shown below."
                        )
                elif unconfirmed_boundaries:
                    boundary_frame, previous_id, new_id = unconfirmed_boundaries[0]
                    SELECTION_FRAME_OVERRIDES[scene.index] = boundary_frame / scene.fps
                    print(
                        f"Unconfirmed Track-ID boundary at "
                        f"{boundary_frame / scene.fps:.2f}s: Track {previous_id} → "
                        f"Track {new_id}. Select the correct visible person at this "
                        "exact boundary; all tracks are shown."
                    )
                elif unsafe_ranges:
                    gap_start, gap_end = max(unsafe_ranges, key=lambda value: value[1] - value[0])
                    gap_midpoint = (gap_start + gap_end - 1) / 2
                    SELECTION_FRAME_OVERRIDES[scene.index] = gap_midpoint / scene.fps
                    print(
                        "Reviewing the longest unresolved interval: "
                        f"{gap_start / scene.fps:.2f}s–{gap_end / scene.fps:.2f}s. "
                        "The next preview is taken from its midpoint."
                    )
            selected, reviewed_frames = await request_manual_anchor(
                scene, scene_records, scene_position,
                allow_occlusion=result is not None and bool(long_unresolved_ranges(scene, result)),
                candidate_track_ids=None,
            )
            if selected == SKIP_SCENE:
                scene_actions[scene.index] = SKIP_SCENE
                result = None
                manual_required = False
            elif selected is None:
                scene_actions[scene.index] = None
                result = None
                manual_required = False
            elif selected == ACCEPT_OCCLUSION:
                result.status = "resolved_with_accepted_occlusion"
                print(
                    "Accepted the unresolved interval as a real occlusion. "
                    "Rendering will hold the last reliable portrait crop; it will not switch to the full frame."
                )
                manual_required = False
            else:
                repairing_existing_resolution = result is not None and bool(result.records)
                print(
                    f"Selection received: Track ID {selected[0]}. "
                    "Building identity evidence…",
                    flush=True,
                )
                face_candidates = []
                for track_id in selected:
                    face_candidates.extend(
                        collect_target_face_candidates(
                            INPUT_VIDEO, scene, scene_records, track_id,
                            face_recognizer, config=resolver_config,
                            anchor_frame=reviewed_frames.get(track_id),
                            progress_description=f"Checking selected Track ID {track_id}",
                        )
                    )
                approved_observations = []
                existing_scene_faces = sum(
                    entry.scene == scene.index for entry in target_identity.entries
                )
                remaining_scene_slots = max(
                    0, MAXIMUM_GALLERY_ENTRIES_PER_SCENE - existing_scene_faces
                )
                raw_face_candidate_count = len(face_candidates)
                if target_identity.ready:
                    face_candidates = [
                        item
                        for item in face_candidates
                        if max(target_identity.similarities(item.embedding), default=-1.0)
                        < GALLERY_DIVERSITY_SIMILARITY
                    ]
                if remaining_scene_slots == 0:
                    face_candidates = []
                if not face_candidates:
                    if raw_face_candidate_count:
                        print(
                            "No gallery review is needed: these faces are already "
                            "represented, or this scene has reached its gallery limit. "
                            "The manual Track ID remains authoritative."
                        )
                    else:
                        print(
                            f"No usable face appeared within {MANUAL_FACE_SEARCH_SECONDS:.2f}s "
                            "of the confirmed frame. Nothing was added to the face gallery; "
                            "the manual Track ID remains authoritative."
                        )
                else:
                    matching_candidates = (
                        [
                            item
                            for item in face_candidates
                            if target_identity.similarity(item.embedding)
                            >= GALLERY_AUTO_ACCEPT_SIMILARITY
                        ]
                        if target_identity.ready
                        else []
                    )
                    coherent_candidates = select_coherent_face_cluster(
                        matching_candidates,
                        similarity_threshold=BOOTSTRAP_FACE_CONSISTENCY_THRESHOLD,
                        maximum_entries=min(
                            resolver_config.bootstrap_gallery_entries,
                            remaining_scene_slots,
                        ),
                        anchor_frame=reviewed_frames.get(selected[0]),
                    )
                    auto_accept_gallery = (
                        target_identity.ready
                        and len(coherent_candidates) >= GALLERY_AUTO_ACCEPT_MINIMUM_FACES
                        and min(item.quality for item in coherent_candidates)
                        >= GALLERY_AUTO_ACCEPT_MINIMUM_QUALITY
                    )
                    if auto_accept_gallery:
                        approved_observations = coherent_candidates
                        print(
                            "Accepted a coherent high-quality face batch that matches "
                            "the already confirmed gallery."
                        )
                    else:
                        approved_observations = await review_face_gallery_candidates_widget(
                            INPUT_VIDEO, face_candidates, target_identity,
                            maximum_selections=remaining_scene_slots,
                        )
                    if approved_observations:
                        added = target_identity.add_many(
                            approved_observations,
                            scene_index=scene.index,
                            source="user_confirmed_gallery",
                        )
                        print(f"Added {added} new trusted face(s) to the gallery.")
                    else:
                        print(
                            "The selected Track ID is still authoritative, but no "
                            "new face was trusted."
                        )
                if target_identity.ready:
                    target_identity.save(GALLERY_ARRAY_FILE, GALLERY_METADATA_FILE)
                    print("Target gallery:", target_identity.public_summary())
                    show_target_gallery(INPUT_VIDEO, target_identity)
                    repaired_fragment = IdentityResolver(
                        face_recognizer, target_identity, resolver_config
                    ).resolve_scene(
                        INPUT_VIDEO, scene, scene_records,
                        anchor_track_id=selected[0],
                        anchor_frame=reviewed_frames.get(selected[0]),
                        resolve_backward=not repairing_existing_resolution,
                        progress=True,
                    )
                    result = supplement_resolution(
                        scene, result, repaired_fragment,
                        maximum_unverified_seconds=MAXIMUM_UNVERIFIED_SECONDS,
                        maximum_confirmation_lead_frames=TRACKING_FRAME_STRIDE,
                    )
                else:
                    print("WARNING: no usable target face was found; using manual Track IDs for this scene only.")
                    manual_fragment = manual_resolution_from_tracks(
                        scene, scene_records, selected,
                        start_frame=(
                            reviewed_frames.get(selected[0])
                            if repairing_existing_resolution
                            else None
                        ),
                    )
                    result = supplement_resolution(
                        scene, result, manual_fragment,
                        maximum_unverified_seconds=MAXIMUM_UNVERIFIED_SECONDS,
                        maximum_confirmation_lead_frames=TRACKING_FRAME_STRIDE,
                    )
                    if (
                        not long_unresolved_ranges(scene, result)
                        and not unconfirmed_source_switches(
                            result,
                            maximum_confirmation_lead_frames=TRACKING_FRAME_STRIDE,
                        )
                    ):
                        result.status = "resolved_manual"
                scene_actions[scene.index] = PERSISTENT_TARGET_TRACK_ID
                manual_required = result.status not in {
                    "resolved", "resolved_manual", "resolved_with_accepted_occlusion"
                }
                if manual_required:
                    remaining_switches = unconfirmed_source_switches(
                        result,
                        maximum_confirmation_lead_frames=TRACKING_FRAME_STRIDE,
                    )
                    if remaining_switches:
                        print(
                            "A Track-ID boundary still needs explicit confirmation. "
                            "The next selector will open at that boundary."
                        )
                    else:
                        print(
                            "The target is still unresolved for too long. "
                            "Review the gap preview and choose another visible fragment, "
                            "or mark the target hidden if appropriate."
                        )

        if result is not None:
            identity_results[scene.index] = result
            scene_actions[scene.index] = PERSISTENT_TARGET_TRACK_ID
            print_identity_summary(scene, result)
            if target_identity.ready:
                identity_metadata = build_identity_cache_metadata(
                    video_info, scene, tracking_metadata=scene_tracking_metadata,
                    face_model_metadata=FACE_MODEL_METADATA,
                    resolver_config=resolver_config,
                    gallery_fingerprint=target_identity.fingerprint,
                )
                save_identity_resolution(
                    result,
                    IDENTITY_CACHE_DIRECTORY / f"scene_{scene.index:03d}.json",
                    metadata=identity_metadata,
                )
        SCENE_ACTIONS_FILE.write_text(
            json.dumps({str(key): value for key, value in scene_actions.items()}, indent=2),
            encoding="utf-8",
        )
finally:
    if tracking_session is not None:
        tracking_session.close()
        tracking_session = None

save_identity_events(identity_results, IDENTITY_EVENTS_FILE)
print(f"\nProcessed {len(scene_actions)}/{len(scenes)} scenes.")


## 8. Validate persistent identity coverage


In [ ]:
unfinished = [scene.index for scene in scenes if scene.index not in scene_actions]
if unfinished:
    raise ValueError(f"Unfinished scenes: {unfinished}")

long_gaps = []
unconfirmed_switches = []
for scene in scenes:
    result = identity_results.get(scene.index)
    if result is None:
        continue
    if result.status != "resolved_with_accepted_occlusion":
        for start, end in result.unresolved_ranges:
            duration = (end - start) / scene.fps
            if duration > MAXIMUM_UNVERIFIED_SECONDS:
                long_gaps.append((scene.index, start / scene.fps, end / scene.fps, duration))
    for frame, previous_id, new_id in unconfirmed_source_switches(
        result,
        maximum_confirmation_lead_frames=TRACKING_FRAME_STRIDE,
    ):
        unconfirmed_switches.append(
            (scene.index, frame / scene.fps, previous_id, new_id)
        )

IDENTITY_COVERAGE_SAFE = not long_gaps and not unconfirmed_switches
if long_gaps:
    details = "\n".join(
        f"- Scene {scene}: {start:.2f}s–{end:.2f}s ({duration:.2f}s)"
        for scene, start, end, duration in long_gaps
    )
    print(
        "WARNING: persistent target identity still has long unresolved intervals.\n"
        + details
        + "\nRerun Section 7 to review them; crop planning will remain blocked."
    )
else:
    print("Identity coverage is safe for crop planning.")

if unconfirmed_switches:
    details = "\n".join(
        f"- Scene {scene} at {seconds:.2f}s: Track {previous_id} → Track {new_id}"
        for scene, seconds, previous_id, new_id in unconfirmed_switches
    )
    print(
        "WARNING: rendering is blocked because these Track-ID changes were not "
        "confirmed at their boundaries:\n" + details + "\nRerun Section 7."
    )

resolved_tracking_records = merge_resolved_records(identity_results)
TARGET_TRACK_BY_SCENE = {
    scene.index: scene_actions[scene.index]
    for scene in scenes
}


## 9. Review identity decisions


In [ ]:
for scene in scenes:
    result = identity_results.get(scene.index)
    if result is None:
        print(f"Scene {scene.index}: action={scene_actions[scene.index]}")
        continue
    print_identity_summary(scene, result)
    show_identity_events(INPUT_VIDEO, scene, result, tracking_records, maximum_events=9)


## 10. Plan the stable 9:16 crop from resolved target boxes


In [ ]:
if RENDER_QUALITY not in {"fast", "high"}:
    raise ValueError("RENDER_QUALITY must be 'fast' or 'high'")
if not IDENTITY_COVERAGE_SAFE:
    raise ValueError("Identity review is incomplete. Resolve the intervals reported in Section 8 first.")
render_width = DRAFT_OUTPUT_WIDTH if RENDER_QUALITY == "fast" else OUTPUT_WIDTH
render_height = DRAFT_OUTPUT_HEIGHT if RENDER_QUALITY == "fast" else OUTPUT_HEIGHT
render_padding_mode = "average" if RENDER_QUALITY == "fast" else PADDING_MODE
crop_config = CropConfig(
    output_width=render_width, output_height=render_height,
    horizontal_margin=HORIZONTAL_MARGIN, top_margin=TOP_MARGIN,
    bottom_margin=BOTTOM_MARGIN, smoothing_seconds=SMOOTHING_SECONDS,
    max_interpolation_seconds=MAX_INTERPOLATION_SECONDS,
    low_confidence_threshold=LOW_CONFIDENCE_THRESHOLD,
    padding_mode=render_padding_mode, render_quality=RENDER_QUALITY,
    missing_track_policy=MISSING_TRACK_POLICY,
    absent_scene_policy=ABSENT_SCENE_POLICY,
)
crop_plans, warnings = build_crop_plan(
    video_info, scenes, resolved_tracking_records,
    TARGET_TRACK_BY_SCENE, crop_config,
)
print(f"Render mode: {RENDER_QUALITY} · {render_width}×{render_height} · {render_padding_mode} padding")
print_warning_summary(warnings)


## 11. Preview one planned frame


In [ ]:
from IPython.display import Image, display

preview_path = save_preview(INPUT_VIDEO, crop_plans, video_info, crop_config, PREVIEW_IMAGE)
if preview_path is None:
    print("No preview is available.")
else:
    display(Image(filename=str(preview_path), width=360))


## 12. Render and restore synchronized audio


In [ ]:
rendered_path = render_video(INPUT_VIDEO, SILENT_VIDEO, crop_plans, video_info, crop_config)
source_audio_segments = source_segments_from_plans(crop_plans, video_info.fps)
if ffmpeg_available():
    final_path = mux_original_audio(
        SILENT_VIDEO, INPUT_VIDEO, FINAL_VIDEO,
        start_seconds=processing_range.start_seconds,
        duration_seconds=processing_range.duration_seconds,
        copy_video=True,
        source_segments=source_audio_segments,
    )
else:
    final_path = SILENT_VIDEO
    print("WARNING: FFmpeg is unavailable; the result has no audio.")
print("Final video:", final_path)


## 13. Save reports and play the result


In [ ]:
from IPython.display import FileLink, Video, display

report_path, warnings_path = save_report(
    OUTPUT_DIRECTORY, info=video_info, scenes=scenes,
    config=crop_config, warnings=warnings,
    tracking_metadata={
        **tracking_metadata,
        "persistent_identity": target_identity.public_summary(),
        "identity_events": str(IDENTITY_EVENTS_FILE),
    },
    processing_range={
        "start_frame": processing_range.start_frame,
        "end_frame": processing_range.end_frame,
        "start_seconds": processing_range.start_seconds,
        "end_seconds": processing_range.end_seconds,
        "duration_seconds": processing_range.duration_seconds,
    },
)
final_path = Path(final_path).resolve()
browser_preview_path = create_browser_preview(
    final_path, BROWSER_PREVIEW_VIDEO, width=BROWSER_PREVIEW_WIDTH
)
print("Report:", report_path)
print("Warnings:", warnings_path)
print("Identity events:", IDENTITY_EVENTS_FILE)
display(Video(
    filename=str(browser_preview_path), embed=True,
    width=BROWSER_PREVIEW_WIDTH,
    html_attributes="controls playsinline preload='metadata'",
))
display(FileLink(str(final_path), result_html_prefix="Open or download full video: "))


## Current safeguards and limitations

- Face similarity values are model scores, not calibrated probabilities.
- The gallery is local biometric data. Delete the two gallery files when they
  are no longer required or before changing the target person.
- Automatic gallery expansion is deliberately disabled in this version.
- If faces remain invisible, manual temporary IDs can be used, but long
  unresolved intervals block rendering rather than producing a frozen video.
- Body ReID is not part of version 1.
